## Learning Advanced RAG

### PDF to Markdown

In [ ]:
#import pymupdf4llm
#md = pymupdf4llm.to_markdown("guidelines_ultraclean.pdf", header=False, footer=False)
#import pathlib
#pathlib.Path("ragdata/guidelines_markdown.md").write_bytes(md.encode())

# Markdown is then cleaned manually after generated, to remove any remaining formatting issues and ensure it is in a consistent format for training the LLM.

## RAG SYSTEM EXPANSION

### Import

In [1]:
import pandas as pd
import re
from collections import Counter

### Chunker

#### PDF (in markdown) chunking

In [2]:
with open("rag_data/guidelines_markdown.md", "r", encoding="utf-8") as file:
    markdown_file = file.read()

current_content = []
chunks = []
current_type = None
current_category = None

for line in markdown_file.splitlines():
    if line.startswith("## "):
        if current_type is not None:
            chunks.append({
                "type": current_type,
                "category": current_category,
                "content": "\n".join(current_content).strip()
            })
        current_category = None
        current_type = "intro"
        current_content = [line]
    elif line.startswith("### "):
        if re.match(r"### [a-z]\.", line):  # Check if it's not a category of questionnaire (starts with lowercase after ###)
            current_content.append(line)
        else:
            if current_type is not None:
                chunks.append({
                    "type": current_type,
                    "category": current_category,
                    "content": "\n".join(current_content).strip()
                })
            current_type = "category"
            current_category = line[4:].strip()
            current_content = [line]
    elif line.startswith("#### "):
        chunks.append({
            "type": current_type,
            "category": current_category,
            "content": "\n".join(current_content).strip()
        })
        current_type = "question"
        current_content = [line]
    elif line.startswith("##### "):
        current_content.append(line)
    else:
        current_content.append(line)

chunks.append({
    "type": current_type,
    "category": current_category,
    "content": "\n".join(current_content).strip()
})

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 147


In [6]:
type_counts = Counter(c["type"] for c in chunks)
print(type_counts)

Counter({'question': 122, 'intro': 17, 'category': 8})


#### CSV Appendix1 chunking

In [7]:
df_appendix1 = pd.read_csv('rag_data/appendix1_questionnairemasterandscoring.csv', sep=';', encoding='latin-1', dtype={'no': str})
df_appendix1['answer'] = df_appendix1['answer'].astype(str).str.replace('?', '≤', regex=False)
df_appendix1.head()

,category,no,indicator_code,criteria,answer,max_score,calculated_score,evidence_required,colored
0,Setting and Infrastructure (SI),1.1,NaN,Types of higher education institution,[1] Comprehensive,NaN,NaN,No,NaN
1,Setting and Infrastructure (SI),1.1,NaN,Types of higher education institution,[2] Specialized higher education institution,NaN,NaN,No,NaN
2,Setting and Infrastructure (SI),1.2,NaN,Climate,[1] Tropical wet,NaN,NaN,No,NaN
3,Setting and Infrastructure (SI),1.2,NaN,Climate,[2] Tropical wet and dry,NaN,NaN,No,NaN
4,Setting and Infrastructure (SI),1.2,NaN,Climate,[3] Semiarid,NaN,NaN,No,NaN


In [8]:
csv_appendix1_chunks = []

for no, group in df_appendix1.groupby('no', sort=False):
    text_content = f"""Question {no} — {group.iloc[0]['criteria']}
Category: {group.iloc[0]['category']}
Evidence Required: {group.iloc[0]['evidence_required']}
"""
    indicator_code = group.iloc[0]['indicator_code'] if not pd.isna(group.iloc[0]['indicator_code']) else 'Not Available'
    max_score = group.iloc[0]['max_score'] if not pd.isna(group.iloc[0]['max_score']) else 'Not Available'
    colored = group.iloc[0]['colored'] if not pd.isna(group.iloc[0]['colored']) else 'Not Available'

    text_content += f"Indicator Code: {indicator_code}\n"
    text_content += f"Max Score: {max_score}\n"
    text_content += f"Colored: {colored}\n"
    text_content += "Options:\n"

    for options in group.itertuples():
        if not pd.isna(options.calculated_score):
            text_content += f"{options.answer} (Calculated score: {options.calculated_score})\n"
        else:
            text_content += f"{options.answer} (Calculated score: Not Available)\n"
        
    max_score = float(group.iloc[0]['max_score']) if not pd.isna(group.iloc[0]['max_score']) else -1.0
    csv_appendix1_chunks.append({
        "content": text_content,
        "metadata": {
            "source": "csv_appendix1",
            "category": group.iloc[0]['category'],
            "question_no": no,
            "evidence_required": group.iloc[0]['evidence_required'],
            "max_score": max_score,
            "colored": colored
        }
    })

# Checking the content of the first and 18th chunk to verify correctness
print("--------------------------------")
print("1st chunk content:")
q1 = next(c for c in csv_appendix1_chunks if c['metadata']['question_no'] == '1.1')
print(q1['content'])
print(q1['metadata'])
print("--------------------------------")
print("18th chunk content:")
q18 = next(c for c in csv_appendix1_chunks if c['metadata']['question_no'] == '1.8')
print(q18['content'])
print(q18['metadata'])
print(f"CSV chunks created: {len(csv_appendix1_chunks)}")

--------------------------------
1st chunk content:
Question 1.1 — Types of higher education institution
Category: Setting and Infrastructure (SI)
Evidence Required: No
Indicator Code: Not Available
Max Score: Not Available
Colored: Not Available
Options:
[1] Comprehensive (Calculated score: Not Available)
[2] Specialized higher education institution (Calculated score: Not Available)

{'source': 'csv_appendix1', 'category': 'Setting and Infrastructure (SI)', 'question_no': '1.1', 'evidence_required': 'No', 'max_score': -1.0, 'colored': 'Not Available'}
--------------------------------
18th chunk content:
Question 1.8 — The ratio of open space area to total area
Category: Setting and Infrastructure (SI)
Evidence Required: Yes
Indicator Code: SI1
Max Score: 200.0
Colored: Not Available
Options:
[1] ≤ 1% (Calculated score: 0.05x200)
[2] > 1 - 80% (Calculated score: 0.25x200)
[3] > 80 - 90% (Calculated score: 0.50x200)
[4] > 90 - 95% (Calculated score: 0.75x200)
[5] > 95% (Calculated score

#### CSV Appendix2 chunking

In [17]:
df_appendix2 = pd.read_csv('rag_data/appendix2_listofgreenbuildingelements.csv', sep=';', encoding='latin-1', dtype={'no': str})
df_appendix2.head()

,element_category,gbi_non-residential_existing_building_category,gbi_non-residential_existing_building_element,gbi_non-residential_new_construction_(nrnc)_category,gbi_non-residential_new_construction_(nrnc)_element
0,Element 1. Energy Efficiency,Design & Performance,Minimum EE Performance,Design,Minimum EE Performance
1,Element 1. Energy Efficiency,Design & Performance,Lighting Zoning,Design,Lighting Zoning
2,Element 1. Energy Efficiency,Design & Performance,Electrical Sub-metering,Design,Electrical Sub-metering
3,Element 1. Energy Efficiency,Design & Performance,Renewable Energy,Design,Renewable Energy
4,Element 1. Energy Efficiency,Design & Performance,Advanced or Improved EE Performance - BEI,Design,Advanced EE Performance - BEI


In [50]:
csv_appendix2_chunks = []

for no, group in df_appendix2.groupby('element_category', sort=False):
    text_content = f"""
Category: {group.iloc[0]['element_category']}
"""
    text_content += "Existing building category:\n"
    for sub_categories, element in group.loc[:, ["gbi_non-residential_existing_building_category", "gbi_non-residential_existing_building_element"]].itertuples(index=False):
        sub_categories = sub_categories if not pd.isna(sub_categories) else 'Sub-category Not Available'
        element = element if not pd.isna(element) else 'Element Not Available'
        text_content += f"{sub_categories} | {element}\n"

    text_content += "\nNew construction category:\n"
    for sub_categories, element in group.loc[:, ["gbi_non-residential_new_construction_(nrnc)_category", "gbi_non-residential_new_construction_(nrnc)_element"]].itertuples(index=False):
        sub_categories = sub_categories if not pd.isna(sub_categories) else 'Sub-category Not Available'
        element = element if not pd.isna(element) else 'Element Not Available'
        text_content += f"{sub_categories} | {element}\n"
        
    csv_appendix2_chunks.append({
        "content": text_content,
        "metadata": {
            "source": "csv_appendix2",
            "element_category": group.iloc[0]['element_category'],
        }
    })

# Checking the content of the first and 3rd chunk to verify correctness
print("--------------------------------")
print("1st chunk content:")
q1 = next(c for c in csv_appendix2_chunks if c['metadata']['element_category'] == 'Element 1. Energy Efficiency')
print(q1['content'])
print(q1['metadata'])
print("--------------------------------")
print("3rd chunk content:")
q3 = next(c for c in csv_appendix2_chunks if c['metadata']['element_category'] == 'Element 3. Sustainable Site Planning & Management')
print(q3['content'])
print(q3['metadata'])
print(f"CSV chunks created: {len(csv_appendix2_chunks)}")

--------------------------------
1st chunk content:

Category: Element 1. Energy Efficiency
Existing building category:
Design & Performance | Minimum EE Performance
Design & Performance | Lighting Zoning
Design & Performance | Electrical Sub-metering
Design & Performance | Renewable Energy
Design & Performance | Advanced or Improved EE Performance - BEI
Commissioning | Enhanced or Re-commissioning
Commissioning | On-going Post Occupancy Commissioning
Monitoring, Improvement & Maintenance | EE Monitoring & Improvement
Monitoring, Improvement & Maintenance | Sustainable Maintenance

New construction category:
Design | Minimum EE Performance
Design | Lighting Zoning
Design | Electrical Sub-metering
Design | Renewable Energy
Design | Advanced EE Performance - BEI
Commissioning | Enhanced Commissioning
Commissioning | Post Occupancy Commissioning
Verification & Maintenance | EE Verification
Verification & Maintenance | Sustainable Maintenance

{'source': 'csv_appendix2', 'element_category'

#### CSV Appendix3 Chunking

In [51]:
df_appendix3 = pd.read_csv('rag_data/appendix3_listanddescriptionofsmartbuildingrequirements.csv', sep=';', encoding='latin-1', dtype={'no': str})
df_appendix3.head()

,field_code,field_name,requirement_code,requirement_name,description
0,B,Automation,B1,BMS,Presence of Building Management System (BMS)/B...
1,B,Automation,B2,APP,Interactive support for users via APP or onlin...
2,S,Safety,S1,Intruder Alarm System,Intruder alarm system (recommended: interfaced...
3,S,Safety,S2,Fire-fighting,Fire-fighting system (recommended: interfaced ...
4,S,Safety,S3,Video surveillance,Video surveillance system (recommended: interf...


In [55]:
csv_appendix3_chunks = []

for no, group in df_appendix3.groupby('field_code', sort=False):
    text_content = f"""
Field code: {group.iloc[0]['field_code']}
Field category: {group.iloc[0]['field_name']}
"""
    text_content += "Requirements:\n"
    for code, name, description in group.loc[:, ["requirement_code", "requirement_name", "description"]].itertuples(index=False):
        text_content += f"{code} | {name}: {description}\n"
        
    csv_appendix3_chunks.append({
        "content": text_content,
        "metadata": {
            "source": "csv_appendix3",
            "field_code": group.iloc[0]['field_code'],
            "field_name": group.iloc[0]['field_name']
        }
    })


# Checking the content of the first and 3rd chunk to verify correctness
print("--------------------------------")
print("1st chunk content:")
q1 = next(c for c in csv_appendix3_chunks if c['metadata']['field_code'] == 'B')
print(q1['content'])
print(q1['metadata'])
print("--------------------------------")
print("3rd chunk content:")
q5 = next(c for c in csv_appendix3_chunks if c['metadata']['field_code'] == 'I')
print(q5['content'])
print(q5['metadata'])
print(f"CSV chunks created: {len(csv_appendix3_chunks)}")

--------------------------------
1st chunk content:

Field code: B
Field category: Automation
Requirements:
B1 | BMS: Presence of Building Management System (BMS)/Building Information Modelling (BIM)/Building Automation System (BAS)/Facility Management System (FMS) (recommended requirement)
B2 | APP: Interactive support for users via APP or online service

{'source': 'csv_appendix3', 'field_code': 'B', 'field_name': 'Automation'}
--------------------------------
3rd chunk content:

Field code: I
Field category: Indoor environment
Requirements:
I1 | Thermal comfort: Monitoring (recommended: interfaced with BMS) of environmental parameters related to thermo hygrometric comfort (i.e. air temperature, relative humidity, air velocity, etc.)
I2 | Air quality: Monitoring (recommended: interfaced with BMS) of pollutants (i.e. VOC, PM, CO? ...)
I3 | Real-time: Programming and management in real time according to the occupancy profile of the premises (recommended: interfaced with BMS)
I4 | Passi